# 05. Building the federal resource layer

So far, the project has focused on reported Hotline activity and state-level context. Now I want to look at the other side of the question: the resources available to respond to trafficking.

The Office for Victims of Crime directs users to two federal Assistance Listings for its primary anti-trafficking programs:

- `16.320` — Services for Trafficking Victims
- `16.035` — Preventing Trafficking of Girls

I'm using USAspending.gov to collect grant-level information for these programs.

This will not represent every trafficking service available in a state. Many organizations operate without these specific federal awards, and some awards may serve areas beyond the recipient's immediate location.

For that reason, I will treat these grants as a measure of **federally funded anti-trafficking resource presence**, not total service capacity.

In [29]:
import pandas as pd
import requests

from pathlib import Path


# Find the actual project root even if VS Code launches
# the notebook from inside a nested folder.
current_location = Path.cwd().resolve()

possible_roots = [
    current_location,
    *current_location.parents
]

project_dir = next(
    (
        path
        for path in possible_roots
        if path.name == "human-trafficking-resource-gap-analysis"
    ),
    None
)


if project_dir is None:
    raise FileNotFoundError(
        "Could not locate the main project folder."
    )


resource_raw_dir = (
    project_dir
    / "data"
    / "raw"
    / "ovc"
)

resource_raw_dir.mkdir(
    parents=True,
    exist_ok=True
)


clean_dir = (
    project_dir
    / "data"
    / "cleaned"
)

clean_dir.mkdir(
    parents=True,
    exist_ok=True
)


print("Project folder:")
print(project_dir)

print("\nRaw resource folder:")
print(resource_raw_dir)

print("\nCleaned data folder:")
print(clean_dir)

Project folder:
C:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis

Raw resource folder:
C:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\raw\ovc

Cleaned data folder:
C:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\cleaned


## Searching USAspending

I am pulling grant awards associated with the two Assistance Listings identified by OVC.

I am searching a wider time period than just fiscal year 2024 because a grant can begin in an earlier year and still be active during FY2024. After collecting the awards, I will use each award's start and end dates to determine whether its period of performance overlapped FY2024.

The federal fiscal year runs from October 1 through September 30.

In [30]:
usaspending_url = (
    "https://api.usaspending.gov"
    "/api/v2/search/spending_by_award/"
)


search_filters = {
    "award_type_codes": [
        "02",
        "03",
        "04",
        "05"
    ],

    "time_period": [
        {
            "start_date": "2018-10-01",
            "end_date": "2024-09-30"
        }
    ],

    "program_numbers": [
        "16.320",
        "16.035"
    ],

    "agencies": [
        {
            "type": "awarding",
            "tier": "subtier",
            "name": "Office of Justice Programs",
            "toptier_name": "Department of Justice"
        }
    ]
}


fields = [
    "Award ID",
    "Recipient Name",
    "Start Date",
    "End Date",
    "Award Amount",
    "Awarding Agency",
    "Awarding Sub Agency",
    "CFDA Number",
    "Description",
    "Place of Performance State Code",
    "Place of Performance City Code"
]

## Collecting every page of results

USAspending limits the number of awards returned in one response, so I'm paging through the results until the API tells me there are no more records.

This also keeps the collection process reproducible instead of manually downloading and combining search pages.

In [31]:
all_awards = []

page = 1


while True:

    payload = {
        "filters": search_filters,
        "fields": fields,
        "page": page,
        "limit": 100,
        "sort": "Award Amount",
        "order": "desc",
        "subawards": False
    }


    response = requests.post(
        usaspending_url,
        json=payload,
        timeout=60
    )

    response.raise_for_status()

    page_data = response.json()

    results = page_data.get(
        "results",
        []
    )

    all_awards.extend(results)


    print(
        f"Page {page}: "
        f"{len(results)} awards"
    )


    has_next = (
        page_data
        .get("page_metadata", {})
        .get("hasNext", False)
    )


    if not has_next:
        break


    page += 1


print(
    f"\nTotal award records collected: "
    f"{len(all_awards)}"
)

Page 1: 100 awards
Page 2: 100 awards
Page 3: 100 awards
Page 4: 100 awards
Page 5: 100 awards
Page 6: 100 awards
Page 7: 100 awards
Page 8: 100 awards
Page 9: 100 awards
Page 10: 5 awards

Total award records collected: 905


## First look at the awards

Before filtering for FY2024 activity, I want to check the fields returned by USAspending and make sure the records actually belong to the programs I intended to collect.

In [32]:
awards_raw = pd.DataFrame(
    all_awards
)


print(
    f"Rows: {len(awards_raw)}"
)

print(
    f"Columns: {len(awards_raw.columns)}"
)


awards_raw.head(10)

Rows: 905
Columns: 15


,internal_id,Award ID,Recipient Name,Start Date,End Date,Award Amount,Awarding Agency,Awarding Sub Agency,CFDA Number,Description,Place of Performance State Code,Place of Performance City Code,awarding_agency_id,agency_slug,generated_internal_id
0,231636291,15POVC24GK02579HT,YOUTH COLLABORATORY INC,2024-10-01,2029-09-30,6000000.00,Department of Justice,Office of Justice Programs,16.320,THE PURPOSE OF YOUTH COLLABORATORYS NATIONAL (...,PA,PA99003,252,department-of-justice,ASST_NON_15POVC24GK02579HT_015
1,234938705,2019VMBXK052,THE NATIONAL CENTER FOR MISSING AND EXPLOITED ...,2019-10-01,2022-09-30,3500000.00,Department of Justice,Office of Justice Programs,16.320,THE NATIONAL CENTER FOR MISSING AND EXPLOITED ...,VA,VA01000,252,department-of-justice,ASST_NON_2019VMBXK052_015
2,231636257,15POVC24GK00873HT,"ICF INCORPORATED, L.L.C.",2024-10-01,2027-09-30,3499999.00,Department of Justice,Office of Justice Programs,16.320,"ICF INCORPORATED, L.L.C. (ICF) WILL IMPLEMENT ...",VA,VA66672,252,department-of-justice,ASST_NON_15POVC24GK00873HT_015
3,231635732,15POVC23GK00905HT,"ICF INCORPORATED, L.L.C.",2023-10-01,2026-09-30,2999942.00,Department of Justice,Office of Justice Programs,16.320,HUMAN TRAFFICKING (HT) IS A PROBLEM THAT REQUI...,VA,VA66672,252,department-of-justice,ASST_NON_15POVC23GK00905HT_015
4,231634693,15POVC21GK02595HT,"ICF INCORPORATED, L.L.C.",2021-10-01,2025-09-30,2997522.03,Department of Justice,Office of Justice Programs,16.320,HUMAN TRAFFICKING (HT) IS A PROBLEM THAT MUST ...,VA,VA66672,252,department-of-justice,ASST_NON_15POVC21GK02595HT_015
5,234906315,2015VTBXK001,INTERNATIONAL ASSOCIATION OF CHIEFS OF POLICE,2015-10-01,2020-09-30,2601869.07,Department of Justice,Office of Justice Programs,16.320,ENHANCING LAW ENFORCEMENT HUMAN TRAFFICKING TA...,VA,VA01000,252,department-of-justice,ASST_NON_2015VTBXK001_015
6,234950417,2020VTBXK002,INTERNATIONAL ASSOCIATION OF CHIEFS OF POLICE,2020-10-01,2024-09-30,2500000.00,Department of Justice,Office of Justice Programs,16.320,IACP HUMAN TRAFFICKING TRAINING AND TECHNICAL ...,VA,VA01000,252,department-of-justice,ASST_NON_2020VTBXK002_015
7,231634217,15PNIJ24GG01747HT,COLORADO MESA UNIVERSITY,2025-01-01,2027-12-31,2495790.00,Department of Justice,Office of Justice Programs,16.320,THE INTERNATIONAL ORGANIZATION FOR MIGRATION (...,CO,CO31660,252,department-of-justice,ASST_NON_15PNIJ24GG01747HT_015
8,234950444,2020VTBXK033,FREEDOM NETWORK USA,2020-10-01,2023-09-30,2143163.00,Department of Justice,Office of Justice Programs,16.320,FREEDOM NETWORK TRAINING INSTITUTE HOUSING TRA...,DC,DC50000,252,department-of-justice,ASST_NON_2020VTBXK033_015
9,234950150,2020VMBX0005,COUNTY OF ALAMEDA,2021-01-01,2021-12-31,2000000.00,Department of Justice,Office of Justice Programs,16.320,HUMAN EXPLOITATION AND TRAFFICKING (HEAT) WATC...,CA,CA53000,252,department-of-justice,ASST_NON_2020VMBX0005_015


In [33]:
awards_raw.columns.tolist()

['internal_id',
 'Award ID',
 'Recipient Name',
 'Start Date',
 'End Date',
 'Award Amount',
 'Awarding Agency',
 'Awarding Sub Agency',
 'CFDA Number',
 'Description',
 'Place of Performance State Code',
 'Place of Performance City Code',
 'awarding_agency_id',
 'agency_slug',
 'generated_internal_id']

In [34]:
awards_raw["CFDA Number"].value_counts(
    dropna=False
)

CFDA Number
16.320    882
16.035     20
16.834      2
16.830      1
Name: count, dtype: int64

## Preserving the source extract

I am saving the award search exactly as it came back from USAspending before doing any date filtering or aggregation.

In [35]:
raw_award_file = (
    resource_raw_dir
    / "usaspending_anti_trafficking_awards_raw.csv"
)


awards_raw.to_csv(
    raw_award_file,
    index=False
)


print(raw_award_file)

C:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\raw\ovc\usaspending_anti_trafficking_awards_raw.csv


## Which awards overlapped FY2024?

For this project, I care about resources that may have been active during the same period as the 2024 Hotline analysis.

FY2024 ran from October 1, 2023 through September 30, 2024.

An award is considered to overlap that period if:

- it began on or before September 30, 2024, and
- it ended on or after October 1, 2023.

This is a period-of-performance filter. It does not prove that every funded service was continuously available throughout the entire fiscal year.

In [36]:
awards = awards_raw.copy()


awards["Start Date"] = pd.to_datetime(
    awards["Start Date"],
    errors="coerce"
)

awards["End Date"] = pd.to_datetime(
    awards["End Date"],
    errors="coerce"
)

awards["Award Amount"] = pd.to_numeric(
    awards["Award Amount"],
    errors="coerce"
)

awards["CFDA Number"] = (
    awards["CFDA Number"]
    .astype("string")
    .str.strip()
)


# Keep only the two anti-trafficking programs
# identified for this analysis.
programs_used = [
    "16.320",
    "16.035"
]

awards = (
    awards[
        awards["CFDA Number"].isin(programs_used)
    ]
    .copy()
    .reset_index(drop=True)
)


fy2024_start = pd.Timestamp("2023-10-01")
fy2024_end = pd.Timestamp("2024-09-30")


awards["active_during_fy2024"] = (
    (awards["Start Date"] <= fy2024_end)
    &
    (awards["End Date"] >= fy2024_start)
)


fy2024_awards = (
    awards[
        awards["active_during_fy2024"]
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    "Awards in the two selected programs:",
    len(awards)
)

print(
    "Awards overlapping FY2024:",
    len(fy2024_awards)
)

print("\nPrograms remaining:")

display(
    fy2024_awards["CFDA Number"]
    .value_counts()
    .rename_axis("program")
    .reset_index(name="award_records")
)

Awards in the two selected programs: 902
Awards overlapping FY2024: 404

Programs remaining:


,program,award_records
0,16.320,394
1,16.035,10


## Checking award geography

For the state-level resource analysis, I am using the award's primary place of performance where it is available.

That is more relevant to this question than simply using the recipient organization's mailing address.

There is still an important limitation: some programs serve multiple counties, states, or broader regions, so a single state code cannot fully describe every award's service area.

In [37]:
state_counts = (
    fy2024_awards[
        "Place of Performance State Code"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("state_abbr")
    .reset_index(
        name="award_records"
    )
)


state_counts.head(15)

,state_abbr,award_records
0,CA,61
1,NY,40
2,TX,22
3,FL,19
4,IL,18
5,VA,16
6,OH,13
7,GA,12
8,WA,12
9,MA,11


## Distinguishing awards from organizations

One organization can hold more than one federal award, so the number of award records is not the same thing as the number of funded organizations.

I'm calculating both. The organization count will be especially useful later when comparing states.

In [38]:
resource_by_state = (
    fy2024_awards

    .groupby(
        "Place of Performance State Code",
        dropna=False
    )

    .agg(
        federal_awards=(
            "Award ID",
            "nunique"
        ),

        funded_organizations=(
            "Recipient Name",
            "nunique"
        ),

        total_award_value=(
            "Award Amount",
            "sum"
        )
    )

    .reset_index()

    .rename(
        columns={
            "Place of Performance State Code":
            "state_abbr"
        }
    )
)


resource_by_state = (
    resource_by_state
    .sort_values(
        "funded_organizations",
        ascending=False
    )
    .reset_index(drop=True)
)


resource_by_state.head(15)

,state_abbr,federal_awards,funded_organizations,total_award_value
0,CA,61,39,45858236.65
1,NY,40,29,24999211.36
2,IL,18,16,12204928.27
3,FL,19,15,13425766.22
4,TX,22,15,15077734.37
5,VA,16,11,19030142.11
6,MA,11,9,7639716.18
7,WA,12,8,8064456.98
8,NJ,10,8,7106586.33
9,CT,10,7,6954988.15


## Separating the two federal program groups

The two Assistance Listings capture somewhat different types of anti-trafficking activity.

Rather than combining them immediately, I want to see how much of the award set comes from each program.

In [39]:
program_summary = (
    fy2024_awards

    .groupby(
        "CFDA Number",
        dropna=False
    )

    .agg(
        awards=(
            "Award ID",
            "nunique"
        ),

        organizations=(
            "Recipient Name",
            "nunique"
        ),

        award_value=(
            "Award Amount",
            "sum"
        )
    )

    .reset_index()
)


program_summary

,CFDA Number,awards,organizations,award_value
0,16.035,10,10,4.501958e+06
1,16.320,394,281,2.826311e+08


## Saving the FY2024 resource data

I'm keeping both the individual award records and the state-level summary.

The award-level file gives me an audit trail, while the state summary is what I will eventually join to the Hotline data.

In [40]:
award_file = (
    clean_dir
    / "ovc_awards_active_fy2024.csv"
)

state_resource_file = (
    clean_dir
    / "ovc_resources_by_state_fy2024.csv"
)


fy2024_awards.to_csv(
    award_file,
    index=False
)

resource_by_state.to_csv(
    state_resource_file,
    index=False
)


print("Saved award-level data:")
print(award_file)

print("\nSaved state-level resource summary:")
print(state_resource_file)

Saved award-level data:
C:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\cleaned\ovc_awards_active_fy2024.csv

Saved state-level resource summary:
C:\Users\skybo\OneDrive\Documents\human-trafficking-resource-gap-analysis\data\cleaned\ovc_resources_by_state_fy2024.csv
